# INF TC1 - TD5 - Palettes de couleurs et quantification

## Objectif
Déterminer automatiquement une palette de couleurs optimale pour une image donnée :
- utiliser moins de couleurs que l'image originale
- être la plus représentative possible des couleurs de l'image

## Fonctions autorisées (issues du TD4)
- `getPixel`, `setPixel`, `setRegion`, `moyenne`, `distance`, `floodFill`
- Module PIL uniquement (pas NumPy, pas OpenCV)
- À partir de l'étape 6 : `im.quantize()` de PIL

## Imports et fonctions du TD4

In [ ]:
from PIL import Image, ImageFilter
from IPython.display import display, HTML
from math import sqrt

# ---------- Fonctions héritées du TD4 ----------

def getPixel(x, y, px):
    return px[x, y]

def setPixel(x, y, color, px):
    px[x, y] = int(color[0]), int(color[1]), int(color[2])

def setRegion(x, y, w, h, color, px):
    for i in range(int(x), int(x + w)):
        for j in range(int(y), int(y + h)):
            setPixel(i, j, color, px)

def moyenne(corner_x, corner_y, region_w, region_h, px):
    sr, sg, sb = 0, 0, 0
    area = region_w * region_h
    for i in range(int(corner_x), int(corner_x + region_w)):
        for j in range(int(corner_y), int(corner_y + region_h)):
            r, g, b = getPixel(i, j, px)
            sr += r; sg += g; sb += b
    return (int(sr / area), int(sg / area), int(sb / area))

def distance(c1, c2):
    return sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2 + (c1[2]-c2[2])**2)

def conversion_gris(px, W, H):
    for x in range(W):
        for y in range(H):
            r, g, b = px[x, y]
            v = int(0.2989*r + 0.5870*g + 0.1140*b)
            px[x, y] = v, v, v

def somme_matrice(m):
    return sum(sum(row) for row in m)

def convolution(px, W, H, m):
    w, h = len(m), len(m[0])
    wp, hp = (w-1)//2, (h-1)//2
    sm = somme_matrice(m)
    for x in range(wp, W - wp):
        for y in range(hp, H - hp):
            s = 0
            for a in range(-wp, wp+1):
                for b in range(-hp, hp+1):
                    s += px[x+a, y+b][0] * m[a+wp][b+hp]
            v = int(s / sm)
            px[x, y] = v, v, v

print("Fonctions TD4 chargées.")

---
## Étape 1 — Charger une image, lister les couleurs et leur fréquence

In [ ]:
def lister_couleurs(im):
    """
    Retourne un dictionnaire {couleur: fréquence} pour tous les pixels de l'image.
    Args:
        im (Image): image PIL en mode RGB
    Returns:
        dict: {(r,g,b): count}
    """
    px = im.load()
    W, H = im.size
    freq = {}
    for x in range(W):
        for y in range(H):
            c = getPixel(x, y, px)
            if c in freq:
                freq[c] += 1
            else:
                freq[c] = 1
    return freq

# --- Test sur une image simple : image noire avec un carré blanc ---
W, H = 100, 100
im_test = Image.new('RGB', (W, H))
px_test = im_test.load()
setRegion(W//3, H//3, W//3, H//3, (255, 255, 255), px_test)

freq_test = lister_couleurs(im_test)
print("Couleurs uniques:", len(freq_test))
for c, n in sorted(freq_test.items(), key=lambda x: -x[1]):
    print(f"  {c} → {n} pixels")

assert len(freq_test) == 2, "Il doit y avoir exactement 2 couleurs"
assert freq_test[(0, 0, 0)] + freq_test[(255, 255, 255)] == W * H
print("✓ Test réussi")

In [ ]:
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
print(f"Taille image : {im.size}")
print(f"Nombre de couleurs uniques : {len(freq)}")
top10 = sorted(freq.items(), key=lambda x: -x[1])[:10]
print("Top 10 couleurs les plus fréquentes :")
for c, n in top10:
    print(f"  {c} → {n} pixels")
display(im)

---
## Étape 2 — Méthode naïve de choix d'une palette de k couleurs

**Méthode naïve** : prendre les `k` couleurs les plus fréquentes dans l'image.

In [ ]:
def palette_naive(freq, k):
    """
    Retourne une palette de k couleurs = les k couleurs les plus fréquentes.
    Args:
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées
    Returns:
        list: liste de k tuples (r, g, b)
    """
    # Trier par fréquence décroissante et prendre les k premières
    couleurs_triees = sorted(freq.items(), key=lambda x: -x[1])
    return [c for c, _ in couleurs_triees[:k]]


def afficher_palette(palette, taille_case=50):
    """
    Crée et affiche une image PIL montrant les couleurs de la palette.
    Args:
        palette (list): liste de tuples (r, g, b)
        taille_case (int): taille en pixels de chaque case de couleur
    Returns:
        Image: image PIL de la palette
    """
    k = len(palette)
    im_palette = Image.new('RGB', (k * taille_case, taille_case))
    px_p = im_palette.load()
    for i, couleur in enumerate(palette):
        setRegion(i * taille_case, 0, taille_case, taille_case, couleur, px_p)
    return im_palette


# --- Test sur image 1 couleur (noire) ---
im_1 = Image.new('RGB', (50, 50))  # image entièrement noire
freq_1 = lister_couleurs(im_1)
palette_1 = palette_naive(freq_1, 1)
print("Palette image noire (k=1):", palette_1)
assert palette_1 == [(0, 0, 0)], "Doit retourner [(0,0,0)]"
display(afficher_palette(palette_1))

# --- Test sur image 4 couleurs ---
im_4 = Image.new('RGB', (100, 100))
px_4 = im_4.load()
setRegion(0, 0, 50, 50, (255, 0, 0), px_4)    # rouge
setRegion(50, 0, 50, 50, (0, 255, 0), px_4)   # vert
setRegion(0, 50, 50, 50, (0, 0, 255), px_4)   # bleu
setRegion(50, 50, 50, 50, (255, 255, 0), px_4) # jaune

freq_4 = lister_couleurs(im_4)
palette_4 = palette_naive(freq_4, 4)
print("Palette image 4 couleurs (k=4):", sorted(palette_4))
assert len(palette_4) == 4
display(afficher_palette(palette_4))
display(im_4)

print("✓ Tests étape 2 réussis")

---
## Étape 3 — Re-colorier une image avec une palette de k couleurs

In [ ]:
def couleur_la_plus_proche(c, palette):
    """
    Retourne la couleur de la palette la plus proche de c (distance euclidienne).
    Args:
        c (tuple): couleur (r, g, b)
        palette (list): liste de tuples (r, g, b)
    Returns:
        tuple: couleur la plus proche dans la palette
    """
    plus_proche = palette[0]
    dist_min = distance(c, palette[0])
    for couleur in palette[1:]:
        d = distance(c, couleur)
        if d < dist_min:
            dist_min = d
            plus_proche = couleur
    return plus_proche


def recolorier(im, palette):
    """
    Re-colorier chaque pixel de l'image avec la couleur la plus proche dans la palette.
    Args:
        im (Image): image PIL originale
        palette (list): liste de tuples (r, g, b)
    Returns:
        Image: nouvelle image re-coloriée
    """
    W, H = im.size
    im_out = Image.new('RGB', (W, H))
    px_in = im.load()
    px_out = im_out.load()
    for x in range(W):
        for y in range(H):
            c = getPixel(x, y, px_in)
            c_proche = couleur_la_plus_proche(c, palette)
            setPixel(x, y, c_proche, px_out)
    return im_out


# --- Test : image 4 couleurs re-coloriée avec palette de 2 couleurs ---
# On prend rouge et bleu comme palette réduire → vert et jaune iront vers le plus proche
palette_reduite = [(255, 0, 0), (0, 0, 255)]
im_recoloriee = recolorier(im_4, palette_reduite)
print("Re-coloriage avec 2 couleurs (rouge, bleu) :")
display(im_4)         # originale
display(im_recoloriee) # re-coloriée

# --- Test : image noire re-coloriée avec palette de 1 couleur ---
im_1_recoloriee = recolorier(im_1, [(0, 0, 0)])
freq_1r = lister_couleurs(im_1_recoloriee)
assert list(freq_1r.keys()) == [(0, 0, 0)], "L'image doit être entièrement noire"
print("✓ Test image 1 couleur réussi")

# --- Test complet : image 4 couleurs re-coloriée avec palette exacte ---
im_4_exact = recolorier(im_4, palette_4)
freq_exact = lister_couleurs(im_4_exact)
assert len(freq_exact) == 4, "Doit avoir exactement 4 couleurs"
print("✓ Test image 4 couleurs avec palette exacte réussi")

---
## Étape 4 — Validation : différence et score d'erreur global

In [ ]:
def image_difference(im_orig, im_recoloriee):
    """
    Crée une image montrant la différence pixel par pixel entre deux images.
    Chaque canal est la valeur absolue de la différence.
    Args:
        im_orig (Image): image originale
        im_recoloriee (Image): image re-coloriée
    Returns:
        Image: image de différence
    """
    W, H = im_orig.size
    im_diff = Image.new('RGB', (W, H))
    px_o = im_orig.load()
    px_r = im_recoloriee.load()
    px_d = im_diff.load()
    for x in range(W):
        for y in range(H):
            ro, go, bo = getPixel(x, y, px_o)
            rr, gr, br = getPixel(x, y, px_r)
            # Amplifier la différence pour la visualisation
            diff_r = min(255, abs(ro - rr) * 3)
            diff_g = min(255, abs(go - gr) * 3)
            diff_b = min(255, abs(bo - br) * 3)
            setPixel(x, y, (diff_r, diff_g, diff_b), px_d)
    return im_diff


def score_erreur(im_orig, im_recoloriee):
    """
    Calcule le score d'erreur global moyen entre l'image originale et re-coloriée.
    Score = moyenne des distances euclidiennes pixel par pixel.
    Args:
        im_orig (Image): image originale
        im_recoloriee (Image): image re-coloriée
    Returns:
        float: erreur moyenne (entre 0 et ~441 max)
    """
    W, H = im_orig.size
    px_o = im_orig.load()
    px_r = im_recoloriee.load()
    total = 0.0
    for x in range(W):
        for y in range(H):
            c_o = getPixel(x, y, px_o)
            c_r = getPixel(x, y, px_r)
            total += distance(c_o, c_r)
    return total / (W * H)


# --- Test : image noire re-coloriée avec la bonne palette → erreur 0 ---
erreur_0 = score_erreur(im_1, im_1_recoloriee)
print(f"Erreur image noire re-coloriée (attendu 0.0) : {erreur_0:.4f}")
assert erreur_0 == 0.0

# --- Test : image 4 couleurs re-coloriée avec palette exacte → erreur 0 ---
erreur_4_exact = score_erreur(im_4, im_4_exact)
print(f"Erreur image 4 couleurs avec palette exacte (attendu 0.0) : {erreur_4_exact:.4f}")
assert erreur_4_exact == 0.0

# --- Test : image 4 couleurs re-coloriée avec 2 couleurs → erreur > 0 ---
erreur_4_red = score_erreur(im_4, im_recoloriee)
print(f"Erreur image 4 couleurs avec palette de 2 couleurs : {erreur_4_red:.4f}")
assert erreur_4_red > 0

# Affichage de la différence
print("Image de différence (amplifié x3) :")
display(image_difference(im_4, im_recoloriee))
print("✓ Tests étape 4 réussis")

---
## Étape 5 — Amélioration du choix des k couleurs

**Méthode par intervalles** : trier toutes les couleurs (par leur somme r+g+b), diviser en k intervalles et prendre la couleur médiane de chaque intervalle comme représentant.

In [ ]:
def palette_intervalles(freq, k):
    """
    Choisit k couleurs en triant toutes les couleurs (par luminosité approximative r+g+b),
    en divisant en k intervalles et en prenant la couleur du milieu de chaque intervalle.
    Args:
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées
    Returns:
        list: liste de k tuples (r, g, b)
    """
    # Construire la liste des couleurs avec répétitions (pour peser par fréquence)
    # On trie les couleurs uniques par luminosité approx. (r+g+b)
    couleurs_uniques = sorted(freq.keys(), key=lambda c: c[0] + c[1] + c[2])
    n = len(couleurs_uniques)
    if n <= k:
        return couleurs_uniques  # Moins de couleurs que demandé
    
    palette = []
    taille_intervalle = n / k
    for i in range(k):
        debut = int(i * taille_intervalle)
        fin = int((i + 1) * taille_intervalle)
        milieu = (debut + fin) // 2
        palette.append(couleurs_uniques[milieu])
    return palette


def palette_moyenne_intervalles(freq, k):
    """
    Variante : dans chaque intervalle, prend la MOYENNE des couleurs (pondérée par fréquence)
    plutôt que la couleur du milieu. Meilleure représentativité.
    Args:
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées
    Returns:
        list: liste de k tuples (r, g, b)
    """
    couleurs_uniques = sorted(freq.keys(), key=lambda c: c[0] + c[1] + c[2])
    n = len(couleurs_uniques)
    if n <= k:
        return couleurs_uniques
    
    palette = []
    taille_intervalle = n / k
    for i in range(k):
        debut = int(i * taille_intervalle)
        fin = int((i + 1) * taille_intervalle)
        # Calculer la moyenne pondérée par fréquence dans cet intervalle
        sr, sg, sb, total = 0, 0, 0, 0
        for c in couleurs_uniques[debut:fin]:
            f = freq[c]
            sr += c[0] * f
            sg += c[1] * f
            sb += c[2] * f
            total += f
        if total > 0:
            palette.append((int(sr/total), int(sg/total), int(sb/total)))
        else:
            palette.append(couleurs_uniques[(debut + fin) // 2])
    return palette


# --- Tests sur images connues ---
# Image 1 couleur noire
p1_intervalles = palette_intervalles(freq_1, 1)
assert p1_intervalles == [(0, 0, 0)]
print("✓ Palette image noire (k=1) :", p1_intervalles)

# Image 4 couleurs avec k=4 : doit retrouver les 4 couleurs
p4_intervalles = palette_intervalles(freq_4, 4)
print("Palette intervalles (k=4) :", p4_intervalles)
assert len(p4_intervalles) == 4
display(afficher_palette(p4_intervalles))

# Comparaison naive vs intervalles sur image 4 couleurs
im_4_interv = recolorier(im_4, p4_intervalles)
erreur_naive = score_erreur(im_4, recolorier(im_4, palette_naive(freq_4, 4)))
erreur_interv = score_erreur(im_4, im_4_interv)
print(f"Erreur palette naïve (k=4) : {erreur_naive:.4f}")
print(f"Erreur palette intervalles (k=4) : {erreur_interv:.4f}")
print("✓ Tests étape 5 réussis")

---
## Étape 6 — Comparaison avec `quantize` de PIL

À partir de cette étape, l'utilisation de `im.quantize()` de PIL est autorisée.

In [ ]:
def palette_depuis_quantize(im, k):
    """
    Utilise la fonction quantize de PIL pour obtenir une palette de k couleurs.
    Args:
        im (Image): image PIL en mode RGB
        k (int): nombre de couleurs
    Returns:
        list: liste de k tuples (r, g, b)
    """
    im_q = im.quantize(colors=k)
    # Extraire la palette de l'image quantifiée
    palette_brute = im_q.getpalette()  # liste plate [r,g,b, r,g,b, ...]
    palette = []
    for i in range(k):
        r, g, b = palette_brute[3*i], palette_brute[3*i+1], palette_brute[3*i+2]
        palette.append((r, g, b))
    return palette


def comparer_methodes(im, k):
    """
    Compare les trois méthodes de palettes sur une image donnée.
    Affiche les palettes et les scores d'erreur.
    Args:
        im (Image): image PIL
        k (int): nombre de couleurs
    """
    freq = lister_couleurs(im)
    print(f"\n=== Comparaison pour k={k} couleurs ===")
    print(f"Nombre de couleurs dans l'image originale : {len(freq)}")
    
    methodes = [
        ("Naïve (top k)",          palette_naive(freq, k)),
        ("Intervalles médiane",    palette_intervalles(freq, k)),
        ("Intervalles moyenne",    palette_moyenne_intervalles(freq, k)),
        ("PIL quantize",           palette_depuis_quantize(im, k)),
    ]
    
    for nom, palette in methodes:
        im_r = recolorier(im, palette)
        erreur = score_erreur(im, im_r)
        print(f"  {nom:30s} → erreur = {erreur:.4f}")
        display(afficher_palette(palette, taille_case=40))
        display(im_r)


# --- Test sur image 4 couleurs ---
comparer_methodes(im_4, 4)

# --- Test sur image générée avec distribution connue ---
# Image 100x100 avec 3 couleurs réparties inégalement
im_3 = Image.new('RGB', (100, 100))
px_3 = im_3.load()
setRegion(0, 0, 70, 100, (200, 50, 50), px_3)   # rouge dominant (70%)
setRegion(70, 0, 20, 100, (50, 200, 50), px_3)  # vert (20%)
setRegion(90, 0, 10, 100, (50, 50, 200), px_3)  # bleu (10%)
comparer_methodes(im_3, 3)

---
## Étape 7 — Pré-traitement par flou gaussien

Lisser l'image avant d'extraire la palette permet de réduire le bruit et d'obtenir des couleurs plus représentatives.

In [ ]:
def palette_avec_flou(im, k, rayon_flou=2):
    """
    Applique un flou gaussien PIL sur l'image, puis extrait une palette par intervalles.
    Le flou est uniquement utilisé pour choisir la palette — l'image originale est re-coloriée.
    Args:
        im (Image): image PIL originale
        k (int): nombre de couleurs
        rayon_flou (int): rayon du flou gaussien
    Returns:
        tuple: (palette, im_re-coloriée)
    """
    # Appliquer le flou gaussien de PIL
    im_floue = im.filter(ImageFilter.GaussianBlur(radius=rayon_flou))
    freq_floue = lister_couleurs(im_floue)
    palette = palette_moyenne_intervalles(freq_floue, k)
    im_recoloriee = recolorier(im, palette)  # Re-colorier l'originale
    return palette, im_recoloriee


def comparer_avec_sans_flou(im, k):
    """
    Compare la méthode intervalles avec et sans pré-traitement par flou.
    Args:
        im (Image): image PIL
        k (int): nombre de couleurs
    """
    freq = lister_couleurs(im)
    print(f"\n=== Flou gaussien pour k={k} ===")
    
    # Sans flou
    p_sans_flou = palette_moyenne_intervalles(freq, k)
    im_sans_flou = recolorier(im, p_sans_flou)
    e_sans_flou = score_erreur(im, im_sans_flou)
    print(f"  Sans flou      → erreur = {e_sans_flou:.4f}")
    display(afficher_palette(p_sans_flou))
    display(im_sans_flou)
    
    # Avec flou (différents rayons)
    for rayon in [1, 2, 5]:
        p_flou, im_flou = palette_avec_flou(im, k, rayon_flou=rayon)
        e_flou = score_erreur(im, im_flou)
        print(f"  Avec flou r={rayon}  → erreur = {e_flou:.4f}")
        display(afficher_palette(p_flou))
        display(im_flou)


# --- Test sur images connues ---
# Image 4 couleurs : le flou ne devrait pas aider (les couleurs sont uniformes)
comparer_avec_sans_flou(im_4, 4)

# --- Test sur image 3 couleurs avec distribution inégale ---
comparer_avec_sans_flou(im_3, 3)

In [ ]:
# Optionnel : test sur une vraie image
# im = Image.open("lyon.png").convert("RGB")
# comparer_avec_sans_flou(im, 8)
# comparer_methodes(im, 8)

---
## Étape 8 — Amélioration de la distance : espace perceptuel

La distance euclidienne RGB ne correspond pas à la perception humaine.
La formule **CIEDE2000** dans l'espace Lab est plus proche de la perception humaine,
mais ici on utilise une **approximation pondérée de la distance RGB** qui est plus facile
à implémenter sans NumPy et donne de meilleurs résultats visuels que l'euclidienne simple.

Référence : https://en.wikipedia.org/wiki/Color_difference#Euclidean

In [ ]:
def distance_perceptuelle(c1, c2):
    """
    Distance RGB pondérée selon la sensibilité de l'œil humain.
    L'œil est plus sensible au vert, puis au rouge, puis au bleu.
    Formule : sqrt(2*dR² + 4*dG² + 3*dB²)
    Cette formule est une approximation de la distance CIEDE2000 sans passer
    par la conversion Lab (qui nécessiterait des exponentielles et fonctions avancées).
    Args:
        c1 (tuple): couleur (r, g, b)
        c2 (tuple): couleur (r, g, b)
    Returns:
        float: distance perceptuelle approximée
    """
    dr = c1[0] - c2[0]
    dg = c1[1] - c2[1]
    db = c1[2] - c2[2]
    return sqrt(2*dr*dr + 4*dg*dg + 3*db*db)


def couleur_la_plus_proche_perc(c, palette):
    """Version de couleur_la_plus_proche avec distance perceptuelle."""
    plus_proche = palette[0]
    dist_min = distance_perceptuelle(c, palette[0])
    for couleur in palette[1:]:
        d = distance_perceptuelle(c, couleur)
        if d < dist_min:
            dist_min = d
            plus_proche = couleur
    return plus_proche


def recolorier_perceptuel(im, palette):
    """Re-colorier en utilisant la distance perceptuelle."""
    W, H = im.size
    im_out = Image.new('RGB', (W, H))
    px_in = im.load()
    px_out = im_out.load()
    for x in range(W):
        for y in range(H):
            c = getPixel(x, y, px_in)
            c_proche = couleur_la_plus_proche_perc(c, palette)
            setPixel(x, y, c_proche, px_out)
    return im_out


# --- Test : distance euclidienne vs perceptuelle ---
# Cas où l'œil préfère vert : rouge vs vert à égale distance euclidienne de gris
gris = (128, 128, 128)
rouge_leger = (178, 128, 128)   # +50 sur R
vert_leger = (128, 178, 128)    # +50 sur G

d_eucl_r = distance(gris, rouge_leger)
d_eucl_g = distance(gris, vert_leger)
d_perc_r = distance_perceptuelle(gris, rouge_leger)
d_perc_g = distance_perceptuelle(gris, vert_leger)

print("Distance euclidienne :")
print(f"  gris → rouge_léger : {d_eucl_r:.2f}")
print(f"  gris → vert_léger  : {d_eucl_g:.2f}  (égales, comme attendu)")
print("Distance perceptuelle (l'œil est plus sensible au vert) :")
print(f"  gris → rouge_léger : {d_perc_r:.2f}")
print(f"  gris → vert_léger  : {d_perc_g:.2f}  (plus grande, comme attendu)")

assert d_eucl_r == d_eucl_g, "Les distances euclidiennes doivent être égales"
assert d_perc_g > d_perc_r, "L'œil doit être plus sensible au vert"

# Comparaison sur image 4 couleurs
palette_test = palette_intervalles(freq_4, 2)
im_eucl = recolorier(im_4, palette_test)
im_perc = recolorier_perceptuel(im_4, palette_test)
e_eucl = score_erreur(im_4, im_eucl)
e_perc = score_erreur(im_4, im_perc)
print(f"\nErreur (distance euclidienne, k=2) : {e_eucl:.4f}")
print(f"Erreur (distance perceptuelle, k=2) : {e_perc:.4f}")
display(im_eucl)
display(im_perc)
print("✓ Tests étape 8 réussis")

---
## Étape 9 — Optimisation

### Analyse de complexité

| Opération | Complexité actuelle | Optimisation |
|-----------|--------------------|--------------|
| `lister_couleurs` | O(W×H) | ✓ optimal |
| `recolorier` | O(W×H×k) | Cache des résultats |
| `palette_naive` | O(n log n) | ✓ raisonnable |
| `palette_intervalles` | O(n log n) | ✓ raisonnable |

**Optimisation clé** : pour `recolorier`, au lieu de recalculer la couleur la plus proche pour **chaque pixel** (O(W×H×k)), on peut construire un cache `{couleur_originale: couleur_palette}` car beaucoup de pixels partagent la même couleur.

In [ ]:
def recolorier_optimise(im, palette):
    """
    Version optimisée de recolorier avec cache des correspondances couleur→palette.
    Complexité : O(W×H + C×k) au lieu de O(W×H×k)
    où C = nombre de couleurs uniques (C << W×H pour les images naturelles).
    
    Justification : de nombreux pixels partagent la même couleur dans une image.
    Le cache évite de recalculer la couleur la plus proche pour chaque pixel.
    Args:
        im (Image): image PIL originale
        palette (list): liste de tuples (r, g, b)
    Returns:
        Image: nouvelle image re-coloriée
    """
    W, H = im.size
    im_out = Image.new('RGB', (W, H))
    px_in = im.load()
    px_out = im_out.load()
    cache = {}  # {couleur_originale: couleur_palette}
    
    for x in range(W):
        for y in range(H):
            c = getPixel(x, y, px_in)
            if c not in cache:
                cache[c] = couleur_la_plus_proche(c, palette)
            setPixel(x, y, cache[c], px_out)
    return im_out


# --- Vérification : même résultat que la version non optimisée ---
palette_test = palette_intervalles(freq_4, 4)
im_normal = recolorier(im_4, palette_test)
im_opti = recolorier_optimise(im_4, palette_test)

# Vérifier que les deux images sont identiques
px_n = im_normal.load()
px_o = im_opti.load()
W, H = im_4.size
identiques = True
for x in range(W):
    for y in range(H):
        if px_n[x, y] != px_o[x, y]:
            identiques = False
            break
assert identiques, "Les deux versions doivent produire le même résultat"
print("✓ Version optimisée produit le même résultat")

# Mesurer l'impact du cache sur une image 4 couleurs
freq_4 = lister_couleurs(im_4)
print(f"Image 4 couleurs : {W*H} pixels, {len(freq_4)} couleurs uniques")
print(f"Gain théorique du cache : {W*H} appels → {len(freq_4)} appels à distance()")
print(f"Facteur de réduction : {W*H // len(freq_4)}x")

In [ ]:
# --- Mesure de temps (optionnel) ---
import time

# Image un peu plus grande pour voir la différence
im_perf = Image.new('RGB', (200, 200))
px_perf = im_perf.load()
# Créer un dégradé simple pour avoir plus de couleurs uniques
for x in range(200):
    for y in range(200):
        r = (x * 255) // 200
        g = (y * 255) // 200
        b = 100
        px_perf[x, y] = r, g, b

freq_perf = lister_couleurs(im_perf)
palette_perf = palette_intervalles(freq_perf, 8)
print(f"Image test : {im_perf.size}, {len(freq_perf)} couleurs uniques")

t0 = time.time()
im_r1 = recolorier(im_perf, palette_perf)
t1 = time.time()
im_r2 = recolorier_optimise(im_perf, palette_perf)
t2 = time.time()

print(f"Version normale    : {t1-t0:.3f}s")
print(f"Version optimisée  : {t2-t1:.3f}s")
if t1 > t0 and t2 > t1:
    print(f"Accélération       : {(t1-t0)/(t2-t1):.1f}x")

---
## Bonus (Étape 10) — Palette représentative à partir de plusieurs images

In [ ]:
def fusionner_frequences(liste_freq):
    """
    Fusionne plusieurs dictionnaires de fréquences de couleurs en un seul.
    Les fréquences sont additionnées (union pondérée).
    Args:
        liste_freq (list): liste de dicts {couleur: fréquence}
    Returns:
        dict: dictionnaire fusionné {couleur: fréquence_totale}
    """
    freq_total = {}
    for freq in liste_freq:
        for c, n in freq.items():
            if c in freq_total:
                freq_total[c] += n
            else:
                freq_total[c] = n
    return freq_total


def palette_multi_images(liste_images, k):
    """
    Crée une palette représentative à partir de plusieurs images.
    Fusionne les fréquences de couleurs de toutes les images,
    puis applique la méthode par intervalles.
    Args:
        liste_images (list): liste d'images PIL
        k (int): nombre de couleurs
    Returns:
        list: palette de k couleurs représentative de toutes les images
    """
    # Lister les couleurs de chaque image
    liste_freq = [lister_couleurs(im) for im in liste_images]
    # Fusionner toutes les fréquences
    freq_total = fusionner_frequences(liste_freq)
    # Extraire la palette
    return palette_moyenne_intervalles(freq_total, k)


# --- Test : deux images avec des couleurs différentes ---
# Image 1 : rouge/vert
im_a = Image.new('RGB', (100, 50))
px_a = im_a.load()
setRegion(0, 0, 50, 50, (200, 50, 50), px_a)
setRegion(50, 0, 50, 50, (50, 200, 50), px_a)

# Image 2 : bleu/jaune
im_b = Image.new('RGB', (100, 50))
px_b = im_b.load()
setRegion(0, 0, 50, 50, (50, 50, 200), px_b)
setRegion(50, 0, 50, 50, (200, 200, 50), px_b)

palette_multi = palette_multi_images([im_a, im_b], k=4)
print("Palette multi-images (k=4) :", palette_multi)
assert len(palette_multi) == 4
display(afficher_palette(palette_multi))

# Vérification : la palette doit représenter les deux images
e_a = score_erreur(im_a, recolorier_optimise(im_a, palette_multi))
e_b = score_erreur(im_b, recolorier_optimise(im_b, palette_multi))
print(f"Erreur sur image A : {e_a:.4f}")
print(f"Erreur sur image B : {e_b:.4f}")

# Comparer avec palette dédiée à chaque image
e_a_solo = score_erreur(im_a, recolorier_optimise(im_a, palette_moyenne_intervalles(lister_couleurs(im_a), 4)))
e_b_solo = score_erreur(im_b, recolorier_optimise(im_b, palette_moyenne_intervalles(lister_couleurs(im_b), 4)))
print(f"Erreur palette dédiée A : {e_a_solo:.4f}")
print(f"Erreur palette dédiée B : {e_b_solo:.4f}")
print("(La palette multi-images est un compromis entre les deux)")
print("✓ Bonus étape 10 réussi")

---
## Récapitulatif des fonctions

| Étape | Fonction | Description |
|-------|----------|-------------|
| 1 | `lister_couleurs(im)` | Dictionnaire {couleur: fréquence} |
| 2 | `palette_naive(freq, k)` | Top k couleurs les plus fréquentes |
| 2 | `afficher_palette(palette)` | Afficher la palette sous forme d'image |
| 3 | `couleur_la_plus_proche(c, palette)` | Distance euclidienne vers la palette |
| 3 | `recolorier(im, palette)` | Re-colorier image avec palette |
| 4 | `image_difference(im1, im2)` | Visualiser la différence |
| 4 | `score_erreur(im1, im2)` | Score d'erreur global (distance moyenne) |
| 5 | `palette_intervalles(freq, k)` | Intervalles + couleur médiane |
| 5 | `palette_moyenne_intervalles(freq, k)` | Intervalles + moyenne pondérée |
| 6 | `palette_depuis_quantize(im, k)` | PIL quantize |
| 6 | `comparer_methodes(im, k)` | Tableau comparatif |
| 7 | `palette_avec_flou(im, k, rayon)` | Flou gaussien avant extraction |
| 8 | `distance_perceptuelle(c1, c2)` | Distance RGB pondérée |
| 8 | `recolorier_perceptuel(im, palette)` | Re-coloriage perceptuel |
| 9 | `recolorier_optimise(im, palette)` | Cache des correspondances |
| 10 | `fusionner_frequences(liste_freq)` | Fusion de fréquences |
| 10 | `palette_multi_images(liste_images, k)` | Palette multi-images |